# NYC Food Safety Intelligence Dashboard

## Milestone 3 Final Notebook

This notebook turns the earlier milestone work into a more polished, presentation-ready dashboard analysis for New York City restaurant food safety.

### Milestone 3 goals covered here
- strengthen the data-cleaning pipeline
- use meaningful colors that communicate safe vs risky outcomes clearly
- create decision-friendly visualizations for class presentation
- highlight actionable insights for diners, students, and tourists
- export clean summary tables for a future localhost dashboard

### Color logic used throughout
- **Green** = safer / stronger outcome
- **Amber** = caution / medium concern
- **Red** = worse / higher risk
- **Blue** = neutral trend / informational context


## Chunk 1: Import Libraries and Define a Consistent Visual Language

### Why I am doing this
For Milestone 3, I want the dashboard to feel intentional and easy to interpret during presentation. Before doing any analysis, I define the main libraries and a color system that stays consistent across all charts.

### Why this matters for the project
This project is about helping users interpret restaurant safety patterns quickly. If colors change meaning from one chart to another, the dashboard becomes confusing. By defining the palette up front, I make the visuals more trustworthy and easier to explain.


In [6]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

SAFE_GREEN = "#2E8B57"
CAUTION_AMBER = "#E3A008"
ALERT_RED = "#D64545"
INFO_BLUE = "#2563EB"
SLATE = "#475569"
LIGHT_BG = "#F8FAFC"

GRADE_COLORS = {
    "A": SAFE_GREEN,
    "B": CAUTION_AMBER,
    "C": ALERT_RED,
}

RISK_COLORS = {
    "Low": SAFE_GREEN,
    "Medium": CAUTION_AMBER,
    "High": ALERT_RED,
}

PLOT_TEMPLATE = "plotly_white"

print("Libraries and presentation color palette loaded.")


ModuleNotFoundError: No module named 'ipywidgets'

## Chunk 2: Load the Dataset with a Flexible File Path

### Why I am doing this
I want the notebook to work both on my own computer and inside the project workspace. Instead of relying on only one hard-coded path, I check a few possible dataset locations.

### Why this matters for the project
This makes the notebook more reproducible. A professor or teammate can understand exactly where the data is expected to come from, and the notebook is less likely to fail because of a path issue.


In [ ]:
DATA_PATH_OPTIONS = [
    Path("data/nyc_inspection_data.csv.gz"),
    Path("nyc_inspection_data.csv.gz"),
    Path("dashboard_app/data/nyc_inspection_data.csv.gz"),
    Path("nyc_inspection_data.csv"),
    Path("data/nyc_inspection_data.csv"),
]

DATA_PATH = next((path for path in DATA_PATH_OPTIONS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find nyc_inspection_data.csv or nyc_inspection_data.csv.gz. Update DATA_PATH_OPTIONS in this notebook."
    )

raw = pd.read_csv(DATA_PATH)
print(f"Loaded dataset from: {DATA_PATH}")
print("Raw shape:", raw.shape)
raw.head()


## Chunk 3: Clean and Standardize the Inspection Data

### Why I am doing this
The raw NYC inspection file includes inconsistent column names, repeated inspections for the same restaurant, missing values, and non-standard formatting in fields like score, ZIP code, and dates. This chunk standardizes the dataset so the later visuals are based on comparable values.

### Why this matters for the project
Without cleaning, the dashboard could produce misleading results. For example, scores would not be comparable if some records were missing dates or if grades outside A, B, and C were mixed into the same charts. This chunk creates a reliable foundation for every later milestone insight.


In [ ]:
df = raw.copy()
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

rename_map = {
    "boro": "borough",
    "zip": "zipcode",
    "bldg": "building",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

if "inspection_date" in df.columns:
    df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")
if "grade_date" in df.columns:
    df["grade_date"] = pd.to_datetime(df["grade_date"], errors="coerce")
if "record_date" in df.columns:
    df["record_date"] = pd.to_datetime(df["record_date"], errors="coerce")
if "score" in df.columns:
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
if "zipcode" in df.columns:
    df["zipcode"] = (
        df["zipcode"].astype(str).str.extract(r"(\d{5})", expand=False)
    )

text_cols = [
    "dba", "borough", "cuisine_description", "inspection_type",
    "action", "grade", "critical_flag", "violation_description"
]
for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "NaN": np.nan})
        )

usable = df.dropna(subset=["inspection_date", "inspection_type"]).copy()
usable = usable[usable["grade"].isin(["A", "B", "C"])].copy()
usable = usable.drop_duplicates()

usable["borough"] = usable["borough"].fillna("Unknown")
usable["cuisine_description"] = usable["cuisine_description"].fillna("Unknown")
usable["dba"] = usable["dba"].fillna("Unknown Restaurant")
usable["year"] = usable["inspection_date"].dt.year
usable["month"] = usable["inspection_date"].dt.month
usable["year_month"] = usable["inspection_date"].dt.to_period("M").astype(str)

def risk_bucket(score):
    if pd.isna(score):
        return np.nan
    if score <= 13:
        return "Low"
    if score <= 27:
        return "Medium"
    return "High"

usable["risk_level"] = usable["score"].apply(risk_bucket)
usable["has_critical_violation"] = usable["critical_flag"].str.lower().eq("critical")

latest = (
    usable.sort_values(["camis", "inspection_date"], ascending=[True, False])
    .drop_duplicates(subset=["camis"], keep="first")
    .copy()
)

latest["latitude"] = pd.to_numeric(latest["latitude"], errors="coerce")
latest["longitude"] = pd.to_numeric(latest["longitude"], errors="coerce")

map_df = latest[
    latest["latitude"].between(40.45, 40.95)
    & latest["longitude"].between(-74.30, -73.65)
].copy()

print("Cleaned analysis shape:", usable.shape)
print("Latest restaurant snapshot:", latest.shape)
print("Map-ready restaurants:", map_df.shape)
usable.head()


## Data Quality Notes

- The raw file contains repeated inspections for the same restaurant, so the notebook keeps both:
  - a full cleaned inspection table for trends
  - a **latest inspection snapshot** for restaurant-level comparisons
- Grades were limited to **A, B, and C** so the class presentation focuses on interpretable public-facing outcomes.
- Scores are treated as **lower is better**, which drives the green-to-red encoding in summary charts.


## Chunk 4: Build High-Level KPI Metrics for the Dashboard

### Why I am doing this
Before moving into detailed charts, I summarize the overall state of restaurant safety in a few headline metrics. These include how many restaurants are in the latest snapshot, what share earned an A, the average inspection score, and how often critical violations appear.

### Why this matters for the project
This chunk gives the audience a fast overview of the citywide situation. It works well as an opening slide or dashboard header because it frames the rest of the analysis in a concise and presentation-friendly way.


In [ ]:
kpi_restaurants = latest["camis"].nunique()
kpi_grade_a_pct = (latest["grade"].eq("A").mean() * 100)
kpi_avg_score = latest["score"].mean()
kpi_critical_pct = (latest["has_critical_violation"].mean() * 100)

fig_kpi = go.Figure()
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_restaurants,
    title={"text": "Restaurants in Latest Snapshot"},
    number={"font": {"color": SLATE, "size": 36}},
    domain={"row": 0, "column": 0},
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_grade_a_pct,
    title={"text": "A Grade Share (%)"},
    number={"suffix": "%", "font": {"color": SAFE_GREEN, "size": 36}},
    domain={"row": 0, "column": 1},
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_avg_score,
    title={"text": "Average Inspection Score"},
    number={"font": {"color": INFO_BLUE, "size": 36}, "valueformat": ".1f"},
    domain={"row": 0, "column": 2},
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_critical_pct,
    title={"text": "Critical Violation Share (%)"},
    number={"suffix": "%", "font": {"color": ALERT_RED, "size": 36}, "valueformat": ".1f"},
    domain={"row": 0, "column": 3},
))
fig_kpi.update_layout(
    grid={"rows": 1, "columns": 4, "pattern": "independent"},
    template=PLOT_TEMPLATE,
    height=260,
    margin=dict(l=20, r=20, t=50, b=20),
    title="Milestone 3 Dashboard Summary"
)
fig_kpi.show()


## Chunk 5: Compare Boroughs Using Average Inspection Score

### Why I am doing this
One of the most natural questions in a citywide food safety project is whether safety outcomes vary across boroughs. I group restaurants by borough and compare their latest average inspection scores.

### Why this matters for the project
This chunk supports geographic comparison, which is important for the dashboard’s driving question. I also use a green-to-red scale here because lower scores are better, so the color directly reinforces the meaning of the values.


In [ ]:
borough_summary = (
    latest.groupby("borough", as_index=False)
    .agg(
        avg_score=("score", "mean"),
        restaurant_count=("camis", "nunique"),
        critical_rate=("has_critical_violation", "mean"),
    )
)
borough_summary["critical_rate"] = borough_summary["critical_rate"] * 100
borough_summary = borough_summary.sort_values("avg_score", ascending=True)

fig_borough = px.bar(
    borough_summary,
    x="borough",
    y="avg_score",
    color="avg_score",
    color_continuous_scale="RdYlGn_r",
    text="restaurant_count",
    template=PLOT_TEMPLATE,
    title="Average Inspection Score by Borough"
)
fig_borough.update_traces(
    texttemplate="n=%{text}",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Average score: %{y:.2f}<br>"
        "Restaurants: %{text}<extra></extra>"
    )
)
fig_borough.update_layout(
    xaxis_title="Borough",
    yaxis_title="Average Score (Lower = Better)",
    coloraxis_colorbar_title="Avg Score",
    height=520
)
fig_borough.show()

borough_summary


## Chunk 6: Show the Grade Mix by Borough

### Why I am doing this
Average score alone does not tell the full story. To make the borough comparison easier for a broader audience, I also show the grade distribution within each borough.

### Why this matters for the project
Grades are more familiar to most viewers than raw inspection scores. Using green for A, amber for B, and red for C creates an intuitive summary that is especially useful in a live class presentation.


In [ ]:
grade_mix = (
    latest.groupby(["borough", "grade"])
    .size()
    .reset_index(name="count")
)
grade_totals = grade_mix.groupby("borough")["count"].transform("sum")
grade_mix["share_pct"] = grade_mix["count"] / grade_totals * 100

fig_grade_mix = px.bar(
    grade_mix,
    x="borough",
    y="share_pct",
    color="grade",
    barmode="stack",
    color_discrete_map=GRADE_COLORS,
    template=PLOT_TEMPLATE,
    title="Grade Mix by Borough"
)
fig_grade_mix.update_layout(
    xaxis_title="Borough",
    yaxis_title="Share of Restaurants (%)",
    legend_title="Grade",
    height=520
)
fig_grade_mix.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Grade %{legendgroup}: %{y:.1f}%<extra></extra>"
    )
)
fig_grade_mix.show()


## Chunk 7: Track How Inspection Scores Change Over Time

### Why I am doing this
Milestone 3 should go beyond static comparisons and show temporal patterns. This chunk groups inspections by month to examine whether average scores improve, worsen, or remain stable over time.

### Why this matters for the project
A time trend adds depth to the dashboard because it shows that restaurant safety is not just a location-based issue. It also helps answer whether recent conditions look different from earlier periods in the dataset.


In [ ]:
monthly_trend = (
    usable.dropna(subset=["score"])
    .groupby("year_month", as_index=False)
    .agg(
        avg_score=("score", "mean"),
        inspections=("camis", "count")
    )
)
monthly_trend = monthly_trend[monthly_trend["year_month"] >= "2018-01"].copy()

fig_trend = px.line(
    monthly_trend,
    x="year_month",
    y="avg_score",
    markers=True,
    template=PLOT_TEMPLATE,
    title="Average Inspection Score Over Time"
)
fig_trend.update_traces(
    line=dict(color=INFO_BLUE, width=3),
    marker=dict(size=6, color=INFO_BLUE),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Average score: %{y:.2f}<extra></extra>"
    )
)
fig_trend.update_layout(
    xaxis_title="Year-Month",
    yaxis_title="Average Score (Lower = Better)",
    height=520
)
fig_trend.show()


## Chunk 8: Identify Cuisines with Higher Critical Violation Rates

### Why I am doing this
Cuisine categories are an important business and public-health dimension in this dataset. In this chunk, I compare cuisines by critical violation rate, while requiring a minimum number of restaurants so that the comparison is more stable.

### Why this matters for the project
This chunk moves the dashboard from simple description toward pattern discovery. It helps show whether some cuisine groups appear more frequently in higher-risk inspection outcomes, while avoiding over-interpreting very small categories.


In [ ]:
cuisine_summary = (
    latest.groupby("cuisine_description", as_index=False)
    .agg(
        restaurants=("camis", "nunique"),
        avg_score=("score", "mean"),
        critical_rate=("has_critical_violation", "mean"),
    )
)
cuisine_summary["critical_rate"] = cuisine_summary["critical_rate"] * 100
cuisine_summary = cuisine_summary[cuisine_summary["restaurants"] >= 30].copy()

top_cuisines = cuisine_summary.sort_values("critical_rate", ascending=False).head(15)

fig_cuisine = px.bar(
    top_cuisines.sort_values("critical_rate", ascending=True),
    x="critical_rate",
    y="cuisine_description",
    orientation="h",
    color="critical_rate",
    color_continuous_scale=[[0, SAFE_GREEN], [0.5, CAUTION_AMBER], [1, ALERT_RED]],
    text="restaurants",
    template=PLOT_TEMPLATE,
    title="Top 15 Cuisines by Critical Violation Rate"
)
fig_cuisine.update_traces(
    texttemplate="n=%{text}",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Critical violation rate: %{x:.1f}%<br>"
        "Restaurants: %{text}<extra></extra>"
    )
)
fig_cuisine.update_layout(
    xaxis_title="Critical Violation Rate (%)",
    yaxis_title="Cuisine",
    coloraxis_colorbar_title="Critical %",
    height=650
)
fig_cuisine.show()


## Chunk 9: Build an Interactive Restaurant Safety Map with Clean Hover Labels and Filters

### Why I am doing this
An interactive map makes the project feel more like a real dashboard rather than a static report. I use the latest restaurant snapshot with valid latitude and longitude values so users can explore food safety outcomes spatially.

### Why this matters for the project
This is one of the strongest presentation visuals because it combines location, score, grade, cuisine, and risk level in one place. I also add dropdown filters for borough and restaurant type so the dashboard is easier to explore during presentation, and I replace the default hover formatting with cleaner labels that read more like a finished product.


In [ ]:
map_df["restaurant_name"] = map_df["dba"].fillna("Unknown Restaurant")
map_df["cuisine_clean"] = map_df["cuisine_description"].fillna("Unknown").str.title()
map_df["inspection_day"] = map_df["inspection_date"].dt.strftime("%b %d, %Y")
map_df["borough_clean"] = map_df["borough"].fillna("Unknown")

borough_options = ["All Boroughs"] + sorted(map_df["borough_clean"].dropna().unique().tolist())
cuisine_options = ["All Restaurant Types"] + sorted(map_df["cuisine_clean"].dropna().unique().tolist())

borough_dropdown = widgets.Dropdown(
    options=borough_options,
    value="All Boroughs",
    description="Borough:",
    layout=widgets.Layout(width="320px")
)

cuisine_dropdown = widgets.Dropdown(
    options=cuisine_options,
    value="All Restaurant Types",
    description="Type:",
    layout=widgets.Layout(width="360px")
)

output_map = widgets.Output()

def render_map(selected_borough, selected_cuisine):
    filtered = map_df.copy()

    if selected_borough != "All Boroughs":
        filtered = filtered[filtered["borough_clean"] == selected_borough]

    if selected_cuisine != "All Restaurant Types":
        filtered = filtered[filtered["cuisine_clean"] == selected_cuisine]

    with output_map:
        output_map.clear_output(wait=True)

        if filtered.empty:
            print("No restaurants match the selected filters.")
            return

        fig_map = px.scatter_map(
            filtered,
            lat="latitude",
            lon="longitude",
            color="risk_level",
            color_discrete_map=RISK_COLORS,
            custom_data=[
                "restaurant_name",
                "borough_clean",
                "cuisine_clean",
                "grade",
                "score",
                "inspection_day",
                "risk_level",
            ],
            zoom=10,
            height=720,
            title="NYC Restaurant Safety Map (Latest Inspection per Restaurant)"
        )

        fig_map.update_traces(
            marker={"size": 9, "opacity": 0.80},
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Borough: %{customdata[1]}<br>"
                "Restaurant Type: %{customdata[2]}<br>"
                "Grade: %{customdata[3]}<br>"
                "Score: %{customdata[4]:.0f}<br>"
                "Inspection Date: %{customdata[5]}<br>"
                "Risk Level: %{customdata[6]}"
                "<extra></extra>"
            ),
        )

        fig_map.update_layout(
            template=PLOT_TEMPLATE,
            legend_title_text="Risk Level",
            margin=dict(l=10, r=10, t=60, b=10)
        )
        fig_map.show()

def update_from_widgets(change=None):
    render_map(borough_dropdown.value, cuisine_dropdown.value)

borough_dropdown.observe(update_from_widgets, names="value")
cuisine_dropdown.observe(update_from_widgets, names="value")

display(widgets.HBox([borough_dropdown, cuisine_dropdown]))
display(output_map)
render_map("All Boroughs", "All Restaurant Types")


## Chunk 10: Create a “Quick Safer Picks” Table for Real Users

### Why I am doing this
The dashboard should not only describe the data, but also translate it into something practical. This chunk identifies restaurants with an A grade, low risk level, and no critical violation in their latest inspection.

### Why this matters for the project
This chunk shows the applied value of the project. Instead of only discussing patterns, it demonstrates how a diner, tourist, or student could actually use the dashboard to make a safer dining decision.


In [ ]:
quick_picks = latest[
    (latest["grade"] == "A")
    & (latest["risk_level"] == "Low")
    & (~latest["has_critical_violation"])
].copy()

quick_picks = quick_picks.sort_values(
    ["score", "inspection_date"],
    ascending=[True, False]
)

quick_picks_view = quick_picks[
    ["dba", "borough", "cuisine_description", "grade", "score", "inspection_date"]
].head(20).copy()
quick_picks_view["inspection_date"] = quick_picks_view["inspection_date"].dt.strftime("%Y-%m-%d")

fig_table = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[
                    "Restaurant", "Borough", "Cuisine", "Grade", "Score", "Inspection Date"
                ],
                fill_color=SAFE_GREEN,
                font=dict(color="white", size=13),
                align="left",
            ),
            cells=dict(
                values=[
                    quick_picks_view["dba"],
                    quick_picks_view["borough"],
                    quick_picks_view["cuisine_description"],
                    quick_picks_view["grade"],
                    quick_picks_view["score"],
                    quick_picks_view["inspection_date"],
                ],
                fill_color=LIGHT_BG,
                align="left",
                height=28,
            ),
        )
    ]
)
fig_table.update_layout(
    title="Quick Safer Picks for Diners",
    height=650,
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_table.show()


## Chunk 11: Summarize the Most Common Violation Descriptions

### Why I am doing this
To understand what drives inspection problems, I also look directly at the violation text descriptions that appear most often in the cleaned inspection data.

### Why this matters for the project
This chunk helps connect high-level dashboard metrics back to concrete food safety issues. It gives the professor and the audience a clearer sense of what kinds of problems inspectors are repeatedly finding.


In [ ]:
top_violations = (
    usable["violation_description"]
    .dropna()
    .value_counts()
    .head(12)
    .reset_index()
)
top_violations.columns = ["violation_description", "count"]

fig_viol = px.bar(
    top_violations.sort_values("count", ascending=True),
    x="count",
    y="violation_description",
    orientation="h",
    color="count",
    color_continuous_scale=[[0, CAUTION_AMBER], [1, ALERT_RED]],
    template=PLOT_TEMPLATE,
    title="Most Common Violation Descriptions"
)
fig_viol.update_layout(
    xaxis_title="Number of Inspection Records",
    yaxis_title="Violation Description",
    coloraxis_colorbar_title="Count",
    height=680
)
fig_viol.show()


## Milestone 3 Interpretation

### Key takeaways
- Borough score differences are visible, but all boroughs still include both strong and weak restaurant outcomes.
- Grade distributions are easier to explain when color is meaningful: **A = green**, **B = amber**, **C = red**.
- Cuisine comparisons become more honest when filtered to cuisines with enough restaurants to make the comparison stable.
- The map is most useful for presentation because it connects restaurant location, grade, score, and risk in one view.

### Class presentation angle
- Start with the KPI row to summarize the citywide picture.
- Move to boroughs and grades to compare neighborhoods.
- Use the cuisine and violation charts to explain what types of problems appear most often.
- End with the map and safer-picks table to show how the analysis helps real diners make decisions.


## Chunk 12: Export Milestone 3 Tables for Future Dashboard Use

### Why I am doing this
The final step is to save cleaned summary tables that can be reused in a localhost web dashboard or future app version of the project.

### Why this matters for the project
This chunk makes the notebook more than a one-time analysis. It creates structured outputs that can support the next stage of the project, including a browser-based dashboard for presentation.


In [ ]:
output_dir = Path("outputs_milestone3")
output_dir.mkdir(exist_ok=True)

borough_summary.to_csv(output_dir / "borough_summary.csv", index=False)
monthly_trend.to_csv(output_dir / "monthly_trend.csv", index=False)
top_cuisines.to_csv(output_dir / "cuisine_critical_rate.csv", index=False)
quick_picks_view.to_csv(output_dir / "quick_safer_picks.csv", index=False)
map_df[[
    "camis", "restaurant_name", "borough", "cuisine_clean", "grade",
    "score", "risk_level", "inspection_date", "latitude", "longitude"
]].to_csv(output_dir / "map_snapshot.csv", index=False)

print(f"Milestone 3 outputs saved to: {output_dir.resolve()}")


## Chunk 13: Include the Final Dash App Code for the Deliverable

### Why I am doing this
The final class deliverable is a localhost Dash dashboard, so I include the actual application code below in the notebook as part of the final submission package. This keeps the notebook aligned with the live app shown during presentation.

### Why this matters for the project
My professor asked for the final notebook to contain the code needed to run the deliverable, including the Dash app when the deliverable is a dashboard. The same code is also stored in `app.py` for easier local execution, but I include it here so the notebook documents the complete final product.


In [ ]:
from __future__ import annotations

from dash import Dash, Input, Output, dash_table, dcc, html
import plotly.express as px
import plotly.graph_objects as go

from data_utils import apply_filters, load_data_bundle


# ============================================================
# Chunk 1: Visual constants and shared chart settings
# Why this exists:
# This section keeps the dashboard's colors and plotting defaults
# consistent across every chart so the interface feels unified.
# ============================================================
SAFE_GREEN = "#2E8B57"
CAUTION_AMBER = "#D7A229"
ALERT_RED = "#C7522A"
INFO_BLUE = "#2B5F75"
SLATE = "#30404D"
LINE = "#D9D2C3"
CARD = "#FFFDF8"

RISK_COLORS = {"Low": SAFE_GREEN, "Medium": CAUTION_AMBER, "High": ALERT_RED}

PLOT_LAYOUT = dict(
    template="plotly_white",
    paper_bgcolor=CARD,
    plot_bgcolor=CARD,
    font=dict(color=SLATE, family='"Avenir Next", "Helvetica Neue", Arial, sans-serif'),
)


bundle = load_data_bundle()
app = Dash(__name__, title="NYC Food Safety Dashboard")
server = app.server


# ============================================================
# Chunk 2: Reusable UI helper components
# Why this exists:
# These helper functions reduce repetition in the layout and make
# the dashboard easier to maintain and explain.
# ============================================================
def metric_card(title: str, metric_id: str, tone: str) -> html.Div:
    return html.Div(
        className=f"metric-card tone-{tone}",
        children=[
            html.Div(title, className="metric-title"),
            html.Div(id=metric_id, className="metric-value"),
        ],
    )


def filter_dropdown(filter_id: str, label: str, options: list[str], value: str) -> html.Div:
    return html.Div(
        className="filter-field",
        children=[
            html.Label(label),
            dcc.Dropdown(
                id=filter_id,
                options=[{"label": x, "value": x} for x in options],
                value=value,
                clearable=False,
                searchable=True,
                className="clean-dropdown",
            ),
        ],
    )


def chart_card(title: str, graph_id: str, extra_class: str = "") -> html.Section:
    class_name = "chart-card" if not extra_class else f"chart-card {extra_class}"
    return html.Section(
        className=class_name,
        children=[
            html.Div(className="section-header", children=[html.H3(title)]),
            dcc.Graph(id=graph_id, config={"displayModeBar": False}),
        ],
    )


# ============================================================
# Chunk 3: Shared figure typography helper
# Why this exists:
# Instead of styling every figure separately, this helper applies
# a consistent title, axis, legend, and hover style.
# ============================================================
def apply_chart_typography(fig, *, height: int, margin: dict, xaxis_title: str | None = None,
                           yaxis_title: str | None = None, legend_orientation: str = "h"):
    fig.update_layout(
        **PLOT_LAYOUT,
        height=height,
        margin=margin,
        title=dict(
            x=0.02,
            xanchor="left",
            font=dict(
                family='"Palatino Linotype", "Book Antiqua", Georgia, serif',
                size=22,
                color=SLATE,
            ),
        ),
        legend=dict(
            orientation=legend_orientation,
            yanchor="bottom",
            y=1.02 if legend_orientation == "h" else 1,
            xanchor="center" if legend_orientation == "h" else "left",
            x=0.5 if legend_orientation == "h" else 1.02,
            font=dict(size=12, family='"Avenir Next", "Helvetica Neue", Arial, sans-serif'),
            title=dict(font=dict(size=12)),
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=12,
            font_family='"Avenir Next", "Helvetica Neue", Arial, sans-serif',
        ),
    )
    fig.update_xaxes(
        title=xaxis_title,
        title_font=dict(size=13, family='"Avenir Next", "Helvetica Neue", Arial, sans-serif'),
        tickfont=dict(size=11, family='"Avenir Next", "Helvetica Neue", Arial, sans-serif'),
        automargin=True,
    )
    fig.update_yaxes(
        title=yaxis_title,
        title_font=dict(size=13, family='"Avenir Next", "Helvetica Neue", Arial, sans-serif'),
        tickfont=dict(size=11, family='"Avenir Next", "Helvetica Neue", Arial, sans-serif'),
        automargin=True,
    )
    return fig


# ============================================================
# Chunk 4: Dashboard layout
# Why this exists:
# This is the visible structure of the app: title, filters, KPI
# cards, analytical charts, map, and table.
# ============================================================
app.layout = html.Div(
    className="app-shell",
    children=[
        html.Div(className="hero-glow hero-left"),
        html.Div(className="hero-glow hero-right"),
        html.Header(
            className="hero-simple",
            children=[
                html.P("New York Food Safety", className="eyebrow"),
                html.H1("Interactive Inspection Dashboard"),
            ],
        ),
        html.Section(
            className="filters-panel compact-panel",
            children=[
                html.Div(className="section-header", children=[html.H2("Dashboard Filters")]),
                html.Div(
                    className="filters-grid filters-grid-compact",
                    children=[
                        filter_dropdown(
                            "cuisine-filter",
                            "Restaurant Type",
                            bundle.cuisine_options,
                            "All Restaurant Types",
                        ),
                        filter_dropdown(
                            "grade-filter",
                            "Grade",
                            bundle.grade_options,
                            "All Grades",
                        ),
                    ],
                ),
            ],
        ),
        html.Section(
            className="metrics-grid",
            children=[
                metric_card("Restaurants in view", "metric-restaurants", "neutral"),
                metric_card("A grade share", "metric-grade", "safe"),
                metric_card("Average score", "metric-score", "info"),
                metric_card("Critical violation share", "metric-critical", "alert"),
            ],
        ),
        html.Section(
            className="charts-grid",
            children=[
                chart_card("Borough Comparison", "borough-chart", "chart-wide"),
                chart_card("Inspection Trend Over Time", "trend-chart"),
                chart_card("Cuisine Risk Landscape", "cuisine-chart"),
                chart_card("Violation Category Summary", "violation-chart", "chart-wide"),
            ],
        ),
        html.Section(
            className="chart-card map-card",
            children=[
                html.Div(className="section-header", children=[html.H3("NYC Restaurant Safety Map")]),
                html.Div(
                    className="filters-grid map-filter-grid",
                    children=[
                        filter_dropdown(
                            "map-borough-filter",
                            "Map Borough",
                            [option for option in bundle.borough_options if option != "Not Reported"],
                            "All Boroughs",
                        ),
                        filter_dropdown(
                            "map-cuisine-filter",
                            "Map Cuisine",
                            bundle.cuisine_options,
                            "All Restaurant Types",
                        ),
                        html.Div(
                            className="filter-field restaurant-field",
                            children=[
                                html.Label("Restaurant Name"),
                                dcc.Dropdown(
                                    id="map-restaurant-filter",
                                    options=[
                                        {"label": x, "value": x}
                                        for x in bundle.restaurant_options
                                    ],
                                    value="All Restaurants",
                                    clearable=False,
                                    searchable=True,
                                    placeholder="Search restaurant by name",
                                    className="clean-dropdown",
                                ),
                            ],
                        ),
                    ],
                ),
                dcc.Graph(id="map-chart", config={"displayModeBar": False}),
                html.Div(
                    id="inspection-detail-card",
                    className="inspection-detail-card",
                    children=[
                        html.H4("Inspection Comment"),
                        html.P(
                            "Click a restaurant circle on the map to see the full inspection comment here."
                        ),
                    ],
                ),
            ],
        ),
        html.Section(
            className="chart-card",
            children=[
                html.Div(className="section-header", children=[html.H3("Quick Safer Picks")]),
                dash_table.DataTable(
                    id="safer-picks-table",
                    columns=[
                        {"name": "Restaurant", "id": "dba"},
                        {"name": "Borough", "id": "borough"},
                        {"name": "Cuisine", "id": "cuisine_description"},
                        {"name": "Grade", "id": "grade"},
                        {"name": "Score", "id": "score"},
                        {"name": "Inspection Date", "id": "inspection_date"},
                    ],
                    page_size=10,
                    sort_action="native",
                    style_table={"overflowX": "auto"},
                    style_header={
                        "backgroundColor": SAFE_GREEN,
                        "color": "white",
                        "fontWeight": "bold",
                        "border": "none",
                    },
                    style_cell={
                        "backgroundColor": CARD,
                        "color": SLATE,
                        "border": f"1px solid {LINE}",
                        "padding": "10px",
                        "textAlign": "left",
                        "fontFamily": "Georgia, Cambria, serif",
                    },
                ),
            ],
        ),
    ],
)


# ============================================================
# Chunk 5: Main callback for interactivity
# Why this exists:
# Dash uses callbacks to update outputs when a user changes a
# filter or clicks on the map. This is the core interactive
# engine of the dashboard.
# ============================================================
@app.callback(
    Output("metric-restaurants", "children"),
    Output("metric-grade", "children"),
    Output("metric-score", "children"),
    Output("metric-critical", "children"),
    Output("borough-chart", "figure"),
    Output("trend-chart", "figure"),
    Output("cuisine-chart", "figure"),
    Output("violation-chart", "figure"),
    Output("map-chart", "figure"),
    Output("safer-picks-table", "data"),
    Output("inspection-detail-card", "children"),
    Input("cuisine-filter", "value"),
    Input("grade-filter", "value"),
    Input("map-borough-filter", "value"),
    Input("map-cuisine-filter", "value"),
    Input("map-restaurant-filter", "value"),
    Input("map-chart", "clickData"),
)
def update_dashboard(
    cuisine: str,
    grade: str,
    map_borough: str,
    map_cuisine: str,
    map_restaurant: str,
    map_click_data: dict | None,
):
    usable, latest = apply_filters(
        bundle.usable,
        bundle.latest,
        "All Boroughs",
        cuisine,
        grade,
        "All Risk Levels",
    )

    restaurants_count = latest["camis"].nunique()
    grade_a_pct = latest["grade"].eq("A").mean() * 100 if len(latest) else 0
    avg_score = latest["score"].mean() if len(latest) else 0
    critical_pct = latest["has_critical_violation"].mean() * 100 if len(latest) else 0

    # ------------------------------------------------------------
    # Chunk 5A: Borough comparison chart
    # Purpose:
    # Compare boroughs using average score and critical violation
    # rate from the latest restaurant-level inspection snapshot.
    # ------------------------------------------------------------
    borough_summary = (
        latest.groupby("borough_clean", as_index=False)
        .agg(
            avg_score=("score", "mean"),
            restaurant_count=("camis", "nunique"),
            critical_rate=("has_critical_violation", "mean"),
        )
        .sort_values("avg_score", ascending=True)
    )
    borough_summary = borough_summary[borough_summary["borough_clean"] != "Not Reported"].copy()
    borough_summary["critical_rate"] = borough_summary["critical_rate"] * 100

    borough_fig = go.Figure()
    borough_fig.add_trace(
        go.Bar(
            x=borough_summary["borough_clean"],
            y=borough_summary["avg_score"],
            name="Average Inspection Score",
            marker=dict(
                color=INFO_BLUE,
                line=dict(color=CARD, width=1.0),
            ),
            customdata=borough_summary[["restaurant_count", "critical_rate"]],
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Average score: %{y:.2f}<br>"
                "Restaurants: %{customdata[0]:,}<br>"
                "Critical violation rate: %{customdata[1]:.1f}%<extra></extra>"
            ),
        )
    )
    borough_fig.add_trace(
        go.Scatter(
            x=borough_summary["borough_clean"],
            y=borough_summary["critical_rate"],
            name="Critical Violation Rate (%)",
            mode="lines+markers",
            yaxis="y2",
            line=dict(color=INFO_BLUE, width=3),
            marker=dict(size=9, color=INFO_BLUE),
            hovertemplate="<b>%{x}</b><br>Critical violation rate: %{y:.1f}%<extra></extra>",
        )
    )
    borough_fig.update_layout(
        title="Borough Comparison: Score and Critical Violations",
        hovermode="x unified",
        xaxis=dict(title="Borough"),
        yaxis=dict(
            title="Average Inspection Score (Lower = Better)",
            gridcolor=LINE,
            zeroline=False,
        ),
        yaxis2=dict(
            title="Critical Violation Rate (%)",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
    )
    apply_chart_typography(
        borough_fig,
        height=500,
        margin=dict(l=50, r=60, t=78, b=42),
        xaxis_title="Borough",
        yaxis_title="Average Inspection Score (Lower = Better)",
    )

    # ------------------------------------------------------------
    # Chunk 5B: Trend chart
    # Purpose:
    # Show how average inspection score changes over time so the
    # dashboard is not limited to a single static snapshot.
    # ------------------------------------------------------------
    monthly_trend = (
        usable.dropna(subset=["score"])
        .groupby("year_month", as_index=False)
        .agg(avg_score=("score", "mean"), inspections=("camis", "count"))
    )
    monthly_trend = monthly_trend[monthly_trend["year_month"] >= "2018-01"].copy()
    trend_fig = px.line(
        monthly_trend,
        x="year_month",
        y="avg_score",
        markers=True,
        title="Average Inspection Score Over Time",
    )
    trend_fig.update_traces(
        line=dict(color=INFO_BLUE, width=3),
        marker=dict(color=INFO_BLUE, size=6),
        hovertemplate="<b>%{x}</b><br>Average score: %{y:.2f}<extra></extra>",
    )
    apply_chart_typography(
        trend_fig,
        height=500,
        margin=dict(l=40, r=30, t=78, b=62),
        xaxis_title="Year-Month",
        yaxis_title="Average Score (Lower = Better)",
    )

    # ------------------------------------------------------------
    # Chunk 5C: Cuisine landscape chart
    # Purpose:
    # Compare restaurant types using both average score and
    # critical violation rate in a single visual.
    # ------------------------------------------------------------
    cuisine_summary = (
        latest.groupby("cuisine_clean", as_index=False)
        .agg(
            restaurants=("camis", "nunique"),
            avg_score=("score", "mean"),
            critical_rate=("has_critical_violation", "mean"),
        )
    )
    cuisine_summary["critical_rate"] = cuisine_summary["critical_rate"] * 100
    cuisine_summary = cuisine_summary[cuisine_summary["restaurants"] >= 15].copy()
    cuisine_summary = cuisine_summary.sort_values(
        ["critical_rate", "restaurants"], ascending=[False, False]
    ).head(18)

    cuisine_fig = px.scatter(
        cuisine_summary,
        x="avg_score",
        y="critical_rate",
        size="restaurants",
        color="critical_rate",
        hover_name="cuisine_clean",
        color_continuous_scale=[[0, SAFE_GREEN], [0.5, CAUTION_AMBER], [1, ALERT_RED]],
        size_max=38,
        title="Cuisine Risk Landscape",
    )
    cuisine_fig.update_traces(
        marker=dict(line=dict(color=CARD, width=1.2), opacity=0.88),
        hovertemplate=(
            "<b>%{hovertext}</b><br>"
            "Average score: %{x:.2f}<br>"
            "Critical violation rate: %{y:.1f}%<br>"
            "Restaurants: %{marker.size:.0f}<extra></extra>"
        ),
    )
    cuisine_fig.update_layout(coloraxis_colorbar=dict(title="Critical %"))
    apply_chart_typography(
        cuisine_fig,
        height=500,
        margin=dict(l=50, r=40, t=78, b=52),
        xaxis_title="Average Inspection Score",
        yaxis_title="Critical Violation Rate (%)",
    )

    # ------------------------------------------------------------
    # Chunk 5D: Violation category chart
    # Purpose:
    # Summarize which high-level food safety issues appear most
    # often in the inspection records.
    # ------------------------------------------------------------
    category_counts = (
        usable["violation_category"].value_counts().reset_index()
        .rename(columns={"index": "violation_category", "violation_category": "count"})
    )
    category_counts.columns = ["violation_category", "count"]
    total_violation_records = int(category_counts["count"].sum())
    category_counts["share_pct"] = category_counts["count"] / total_violation_records * 100
    category_counts = category_counts.sort_values("count", ascending=True)
    top_violation = category_counts.iloc[-1]

    violation_fig = px.bar(
        category_counts,
        x="count",
        y="violation_category",
        orientation="h",
        color="share_pct",
        color_continuous_scale=[[0.0, SAFE_GREEN], [0.5, CAUTION_AMBER], [1.0, ALERT_RED]],
        text="share_pct",
        title="Violation Category Summary",
    )
    violation_fig.update_traces(
        texttemplate="%{text:.1f}%",
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Inspection records: %{x:,}<br>"
            "Share of inspection records: %{marker.color:.1f}%<extra></extra>"
        ),
    )
    violation_fig.update_layout(
        annotations=[
            dict(
                x=0,
                y=1.12,
                xref="paper",
                yref="paper",
                showarrow=False,
                align="left",
                text=(
                    f"Top issue: <b>{top_violation['violation_category']}</b> "
                    f"with {int(top_violation['count']):,} inspection records "
                    f"({top_violation['share_pct']:.1f}%). "
                    "Records here mean inspection rows, not unique restaurants."
                ),
                font=dict(
                    size=12,
                    color=SLATE,
                    family='"Avenir Next", "Helvetica Neue", Arial, sans-serif',
                ),
            )
        ],
        coloraxis_colorbar=dict(title="Share %"),
    )
    apply_chart_typography(
        violation_fig,
        height=540,
        margin=dict(l=30, r=70, t=110, b=30),
        xaxis_title="Inspection Records",
        yaxis_title="Violation Category",
    )

    # ------------------------------------------------------------
    # Chunk 5E: Interactive map
    # Purpose:
    # Connect location, score, grade, and violation summary in
    # a single exploratory view for restaurant-level decisions.
    # ------------------------------------------------------------
    map_latest = latest[
        latest["latitude"].between(40.45, 40.95)
        & latest["longitude"].between(-74.30, -73.65)
    ].copy()
    map_latest = map_latest[map_latest["borough_clean"] != "Not Reported"]
    if map_borough != "All Boroughs":
        map_latest = map_latest[map_latest["borough_clean"] == map_borough]
    if map_cuisine != "All Restaurant Types":
        map_latest = map_latest[map_latest["cuisine_clean"] == map_cuisine]
    if map_restaurant != "All Restaurants":
        map_latest = map_latest[map_latest["restaurant_name"] == map_restaurant]

    map_fig = px.scatter_map(
        map_latest,
        lat="latitude",
        lon="longitude",
        color="risk_level",
        color_discrete_map=RISK_COLORS,
        custom_data=[
            "restaurant_name",
            "borough_clean",
            "cuisine_clean",
            "grade",
            "score",
            "inspection_day",
            "risk_level",
            "violation_category",
            "violation_description",
            "building",
            "street",
            "zipcode",
        ],
        zoom=9.7,
        height=700,
        title="Latest Restaurant Safety Snapshot Across NYC",
    )
    map_fig.update_traces(
        marker={
            "size": 10,
            "opacity": 0.82,
            "allowoverlap": True,
        },
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Borough: %{customdata[1]}<br>"
            "Cuisine: %{customdata[2]}<br>"
            "Grade: %{customdata[3]}<br>"
            "Score: %{customdata[4]:.0f}<br>"
            "Inspection date: %{customdata[5]}<br>"
            "Risk level: %{customdata[6]}<br>"
            "Violation summary: %{customdata[7]}<extra></extra>"
        ),
    )
    map_fig.update_layout(
        **PLOT_LAYOUT,
        margin=dict(l=10, r=10, t=78, b=10),
        title=dict(
            x=0.02,
            xanchor="left",
            font=dict(
                family='"Palatino Linotype", "Book Antiqua", Georgia, serif',
                size=22,
                color=SLATE,
            ),
        ),
        legend_title_text="Risk Level",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            font=dict(size=12, family='"Avenir Next", "Helvetica Neue", Arial, sans-serif'),
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=12,
            font_family='"Avenir Next", "Helvetica Neue", Arial, sans-serif',
        ),
        mapbox_style="open-street-map",
    )

    # ------------------------------------------------------------
    # Chunk 5F: Click-to-view inspection detail panel
    # Purpose:
    # When a restaurant is clicked on the map, show a cleaner
    # inspection summary panel below the map.
    # ------------------------------------------------------------
    inspection_detail_children = [
        html.H4("Inspection Comment"),
        html.P("Click a restaurant circle on the map to see the full inspection comment here."),
    ]
    if map_click_data and map_click_data.get("points"):
        point = map_click_data["points"][0]
        custom = point.get("customdata", [])
        if len(custom) >= 12:
            address_parts = [str(part).strip() for part in [custom[9], custom[10], custom[11]] if str(part).strip() and str(part).strip().lower() != "nan"]
            address_text = ", ".join(address_parts) if address_parts else "Address not reported"
            inspection_detail_children = [
                html.Div(
                    className="detail-badge-row",
                    children=[
                        html.Span("Selected Restaurant", className="detail-badge"),
                    ],
                ),
                html.H4(custom[0]),
                html.Div(
                    className="detail-meta-grid",
                    children=[
                        html.Div([html.Strong("Borough"), html.Span(custom[1])]),
                        html.Div([html.Strong("Cuisine"), html.Span(custom[2])]),
                        html.Div([html.Strong("Grade"), html.Span(custom[3])]),
                        html.Div([html.Strong("Score"), html.Span(str(custom[4]))]),
                        html.Div([html.Strong("Address"), html.Span(address_text)], className="detail-address-card"),
                    ],
                ),
                html.Div(
                    className="detail-summary-box",
                    children=[
                        html.Span("Violation Summary", className="detail-label"),
                        html.P(custom[7]),
                    ],
                ),
                html.Div(
                    className="detail-comment-box",
                    children=[
                        html.Span("Full Inspection Comment", className="detail-label"),
                        html.P(custom[8]),
                    ],
                ),
            ]

    # ------------------------------------------------------------
    # Chunk 5G: Safer picks table
    # Purpose:
    # Translate analysis into a practical list of restaurants
    # with strong recent safety indicators.
    # ------------------------------------------------------------
    quick_picks = latest[
        (latest["grade"] == "A")
        & (latest["risk_level"] == "Low")
        & (~latest["has_critical_violation"])
    ].copy()
    quick_picks = quick_picks.sort_values(["score", "inspection_date"], ascending=[True, False])
    quick_picks["inspection_date"] = quick_picks["inspection_date"].dt.strftime("%Y-%m-%d")
    table_data = (
        quick_picks[
            ["dba", "borough_clean", "cuisine_clean", "grade", "score", "inspection_date"]
        ]
        .rename(
            columns={
                "borough_clean": "borough",
                "cuisine_clean": "cuisine_description",
            }
        )
        .head(20)
        .to_dict("records")
    )

    return (
        f"{restaurants_count:,}",
        f"{grade_a_pct:.1f}%",
        f"{avg_score:.2f}",
        f"{critical_pct:.1f}%",
        borough_fig,
        trend_fig,
        cuisine_fig,
        violation_fig,
        map_fig,
        table_data,
        inspection_detail_children,
    )


# ============================================================
# Chunk 6: Local app runner
# Why this exists:
# This starts the Dash server on localhost so the app can be
# viewed in the browser during development or presentation.
# ============================================================
if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8050)


## Localhost Note

To launch the final dashboard outside the notebook, run the project from Terminal with `python3 app.py` and then open `http://127.0.0.1:8050`. This is the same codebase included above and stored in the GitHub repository for submission.
